# Devign E01 — Multi-seed

## A. Title and description

**Experiment:** E01 multi-seed. **Goal:** run one Devign training seed for either reproduced baseline or proposed selective routing while keeping the split fixed. **Inputs:** Devign source data, UniXcoder/retrieval assets, the local LLM checkpoint, and optionally a prior run checkpoint. **Outputs:** config, metadata, metrics, predictions, calibration, branch metrics, runtime, compact results, and resumable checkpoints. **LLM:** yes for full inference; reproduced baseline explicitly forces every test record through LLM inspection. **Sessions:** one seed and one configuration at a time; a full inference may span multiple chunked Kaggle sessions.

Run separately: proposed seeds 1, 7, 21, 42, 100; then reproduced_baseline seeds 1, 7, 21, 42, 100. Each notebook execution runs only the selected line.

**Important:** smoke-test results are development checks and must not be used in the paper.


## B. User configuration

Edit only this centralized cell before a run.


In [ ]:
REPOSITORY_URL = "https://github.com/khanhtran0111/VulGuardVN.git"
BRANCH = "camera-ready"
DATASET = "devign"

RUN_MODE = "smoke"       # "smoke", "full", or "dry-run"
SEED = 42
CONFIGURATION = "proposed"

OUTPUT_ROOT = "/kaggle/working/revision_results"
SMOKE_OUTPUT_ROOT = "/kaggle/working/revision_smoke_results"

DEVIGN_INPUT = "/kaggle/input/devign"
SEMANTIC_MODEL_INPUT = "/kaggle/input/unixcoder-base-nine"
AUTO_DOWNLOAD_SEMANTIC_MODEL = False
QWEN_MODEL_INPUT = "/kaggle/input/qwen2-5-coder-7b-instruct-bnb-4bit"

AUTO_DOWNLOAD_MODEL = False
RESUME = True
FORCE_RECLONE = False
RESTORE_CHECKPOINT = False
CHECKPOINT_INPUT = "/kaggle/input/vulguard-devign-e01-checkpoint"
TEST_CHUNK_SIZE = 250       # Set None to disable multi-session inference chunks.
TEST_CHUNK_INDEX = None      # None automatically selects the next unresolved chunk.

# Only used by experiments that read E01 artifacts.
REUSE_E01_RESULTS = True
E01_RESULTS_INPUT = "/kaggle/input/vulguard-devign-e01-results"

SESSION_BUDGET_HOURS = 11.5
MIN_REMAINING_MINUTES = 20
EXPERIMENT = "E01_multiseed"
SPLIT_SEED = 42
DEMO_SEED = 31415
BOOTSTRAP_SEED = 27182
# CONFIGURATION may be "proposed" or "reproduced_baseline".


## C. Kaggle environment checks

Checks working directory, Python, disk, NVIDIA driver, CUDA visibility, GPU name/memory, and UTC start time.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os, platform, shutil, subprocess

KAGGLE_WORKING = Path("/kaggle/working")
ON_KAGGLE = KAGGLE_WORKING.exists() and str(Path.cwd()).startswith("/kaggle")
START_TIME = datetime.now(timezone.utc)
print("Kaggle environment:", ON_KAGGLE)
if not ON_KAGGLE:
    print("WARNING: this notebook is intended for /kaggle/working; use dry-run outside Kaggle.")
print("Python:", platform.python_version())
disk_root = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
usage = shutil.disk_usage(disk_root)
print("Disk GB:", {"total": round(usage.total/2**30, 2), "free": round(usage.free/2**30, 2)})
subprocess.run(["nvidia-smi"], check=False)
try:
    import torch
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory GB:", round(props.total_memory/2**30, 2))
    elif True:
        print("WARNING: this experiment needs GPU for UniXcoder and/or LLM execution.")
except Exception as exc:
    print("WARNING: PyTorch/GPU check failed:", exc)
print("Start time UTC:", START_TIME.isoformat())


## D. Prepare repository

Clone `camera-ready` when absent; otherwise fetch, checkout, and pull the target branch.


In [ ]:
from pathlib import Path
import shutil, subprocess

REPO_DIR = Path("/kaggle/working/VulGuardVN")
if not Path("/kaggle/working").exists() and Path.cwd().name == "VulGuardVN":
    REPO_DIR = Path.cwd()  # local dry-run validation only

if FORCE_RECLONE and REPO_DIR.exists():
    if str(REPO_DIR).startswith("/kaggle/working/"):
        shutil.rmtree(REPO_DIR)
    else:
        raise RuntimeError("FORCE_RECLONE is only allowed under /kaggle/working")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=REPO_DIR, check=True)

CURRENT_BRANCH = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR, text=True).strip()
COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Branch:", CURRENT_BRANCH)
print("Commit SHA:", COMMIT_SHA)
subprocess.run(["git", "status", "--short", "--branch"], cwd=REPO_DIR, check=True)
assert CURRENT_BRANCH == BRANCH
REVISION_DIR = REPO_DIR / "GRACE-improve" / "revision_experiments"


## E. Minimal dependencies

The cell checks imports first and installs only missing packages. It does not upgrade existing TensorFlow, PyTorch, or CUDA packages.


In [ ]:
import importlib, importlib.metadata, importlib.util, subprocess, sys

# Derived from baseline2 imports; install only packages missing from the Kaggle image.
DEPENDENCIES = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
    "joblib": "joblib", "matplotlib": "matplotlib", "dotenv": "python-dotenv",
    "tensorflow": "tensorflow", "torch": "torch", "transformers": "transformers",
    "accelerate": "accelerate", "bitsandbytes": "bitsandbytes", "sentencepiece": "sentencepiece",
    "huggingface_hub": "huggingface-hub",
}
requirement_files = sorted(REPO_DIR.glob("requirements*.txt"))
print("Repository requirement files:", [str(path) for path in requirement_files] or "none; using baseline2 import audit")
missing = [package for module, package in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
print("Missing packages:", missing)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-input", *missing], check=True)

for module in ("numpy", "pandas", "scipy", "sklearn", "joblib", "matplotlib", "dotenv", "tensorflow", "torch", "transformers", "accelerate", "bitsandbytes", "sentencepiece", "huggingface_hub"):
    imported = importlib.import_module(module)
    package = DEPENDENCIES[module]
    try: version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: version = getattr(imported, "__version__", "unknown")
    print(f"{package}={version}")


## F. Repository smoke tests

A failing unit test stops execution before the experiment.


In [ ]:
import subprocess, sys

test_command = [sys.executable, "-m", "unittest", "discover", "-s", "GRACE-improve/revision_experiments/tests", "-p", "test_*.py", "-v"]
print(" ".join(test_command))
subprocess.run(test_command, cwd=REPO_DIR, check=True)


## G. Dry-run and execution

`RUN_MODE='dry-run'` prints and validates exactly one command, then skips execution/output packaging.


In [ ]:
from pathlib import Path
import os, subprocess, sys
sys.path.insert(0, str(REVISION_DIR))
from kaggle_artifacts import next_chunk_index, restore_checkpoint
from kaggle_preflight import prepare_devign_input, prepare_qwen_model, prepare_semantic_model

RESULTS_ROOT = Path(SMOKE_OUTPUT_ROOT if RUN_MODE == "smoke" else OUTPUT_ROOT)
RUN_DIR = RESULTS_ROOT / EXPERIMENT / DATASET / CONFIGURATION / f"seed_{SEED}"
if RUN_MODE != "dry-run":
    DEVIGN_PATH = prepare_devign_input(DEVIGN_INPUT, REPO_DIR)
    SEMANTIC_MODEL_DIR = prepare_semantic_model(SEMANTIC_MODEL_INPUT, REPO_DIR, auto_download=AUTO_DOWNLOAD_SEMANTIC_MODEL)
    QWEN_MODEL_DIR = prepare_qwen_model(QWEN_MODEL_INPUT, REPO_DIR, auto_download=AUTO_DOWNLOAD_MODEL, required=RUN_MODE == "full")
    os.environ["GRACE_RETRIEVAL_MODEL_DIR"] = str(SEMANTIC_MODEL_DIR)
    os.environ["GRACE_AUTO_DOWNLOAD_RETRIEVAL_MODEL"] = str(AUTO_DOWNLOAD_SEMANTIC_MODEL).lower()
    if QWEN_MODEL_DIR is not None: os.environ["GRACE_LOCAL_MODEL_DIR"] = str(QWEN_MODEL_DIR)
if RESTORE_CHECKPOINT and RUN_MODE == "full":
    restored = restore_checkpoint(CHECKPOINT_INPUT, RESULTS_ROOT, dataset=DATASET, experiment=EXPERIMENT, configuration=CONFIGURATION, seed=SEED, commit_sha=COMMIT_SHA)
    assert restored == RUN_DIR
    print("Restored checkpoint:", restored)
RESOLVED_CHUNK_INDEX = TEST_CHUNK_INDEX
if TEST_CHUNK_SIZE is not None and RESOLVED_CHUNK_INDEX is None:
    RESOLVED_CHUNK_INDEX = next_chunk_index(RUN_DIR, TEST_CHUNK_SIZE) if RUN_MODE == "full" else 0
print("Test chunk:", {"size": TEST_CHUNK_SIZE, "index": RESOLVED_CHUNK_INDEX})
runner = REPO_DIR / "GRACE-improve" / "revision_experiments" / "run_revision_experiments.py"
COMMAND = [sys.executable, str(runner), "--dataset", DATASET, "--experiment", EXPERIMENT, "--seed", str(SEED), "--configuration", CONFIGURATION, "--output-directory", str(RESULTS_ROOT)]
COMMAND += ["--split-seed", str(SPLIT_SEED), "--demo-seed", str(DEMO_SEED), "--bootstrap-seed", str(BOOTSTRAP_SEED)]
if RUN_MODE == "smoke": COMMAND += ["--smoke", "--no-resume"]
elif RUN_MODE == "full": COMMAND += (["--resume"] if RESUME else ["--no-resume"])
COMMAND += ["--session-budget-hours", str(SESSION_BUDGET_HOURS), "--min-remaining-minutes", str(MIN_REMAINING_MINUTES), "--session-start-epoch", str(START_TIME.timestamp())]
if TEST_CHUNK_SIZE is not None: COMMAND += ["--test-chunk-size", str(TEST_CHUNK_SIZE), "--test-chunk-index", str(RESOLVED_CHUNK_INDEX)]
if RUN_MODE == "dry-run": COMMAND += ["--dry-run"]
print("Command:", " ".join(COMMAND))
os.environ["GRACE_AUTO_DOWNLOAD_MODEL"] = str(AUTO_DOWNLOAD_MODEL).lower()
subprocess.run(COMMAND, cwd=REPO_DIR, check=True)


## H. Output validation

Required files are checked and JSON files are parsed. Missing values remain missing; they are never inferred.


In [ ]:
import json

if RUN_MODE == "dry-run":
    print("Dry-run complete; output validation is intentionally skipped.")
else:
    required = ('config.json', 'run_metadata.json', 'metrics.json', 'predictions.jsonl', 'calibration.json', 'branch_metrics.json', 'runtime.json')
    missing = [name for name in required if not (RUN_DIR / name).is_file()]
    metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8")) if (RUN_DIR / "run_metadata.json").is_file() else {}
    if missing and metadata.get("status") != "partial": raise FileNotFoundError(f"Missing required outputs: {missing}")
    if missing: print("Partial run; artifacts not produced yet:", missing)
    payloads = {}
    for name in required:
        if name.endswith(".json") and (RUN_DIR / name).is_file():
            payloads[name] = json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
        elif name.endswith(".jsonl") and (RUN_DIR / name).is_file():
            with (RUN_DIR / name).open("r", encoding="utf-8") as handle:
                for line_number, line in enumerate(handle, start=1):
                    if line.strip(): json.loads(line)
            print(f"Validated JSONL: {name}")
    metrics = payloads.get("metrics.json", {})
    calibration = payloads.get("calibration.json", {})
    metadata = payloads.get("run_metadata.json", metadata)
    summary = {
        "dataset": DATASET, "experiment": EXPERIMENT, "configuration": CONFIGURATION, "seed": SEED,
        "status": metadata.get("status"), "sample_count": metrics.get("samples"),
        "accuracy": metrics.get("accuracy"), "precision": metrics.get("precision"), "recall": metrics.get("recall"),
        "f1": metrics.get("f1"), "roc_auc": metrics.get("roc_auc"), "pr_auc": metrics.get("pr_auc"),
        "llm_calls": metrics.get("llm_calls"), "llm_call_ratio": metrics.get("llm_call_ratio"),
        "tau_low": calibration.get("tau_low"), "tau_high": calibration.get("tau_high"), "output_path": str(RUN_DIR),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if metadata.get("status") not in ("complete", "partial"):
        raise RuntimeError(f"Run is neither complete nor resumable partial: {metadata.get('status')!r}")

    if CONFIGURATION == "reproduced_baseline" and RUN_MODE == "full" and metadata.get("status") == "complete":
        assert metrics.get("llm_calls") == metrics.get("samples"), "Baseline did not inspect every test record"
        assert metrics.get("llm_call_ratio") == 1.0, "Baseline LLM-call ratio must be exactly 1.0"
        config_payload = payloads["config.json"]
        assert config_payload.get("force_inspect_all") is True


## I. Package results

A compact result ZIP excludes pipeline caches. If the run is partial, a separate checkpoint ZIP preserves the complete run directory, including `_pipeline`, for the next Kaggle session.


In [ ]:
if RUN_MODE == "dry-run":
    print("Dry-run: no ZIP is created.")
else:
    import sys
    sys.path.insert(0, str(REVISION_DIR))
    from kaggle_artifacts import package_checkpoint, package_run, read_json, write_run_summary
    EXPORTS_DIR = Path("/kaggle/working/exports")
    ZIP_PATH = package_run(RUN_DIR, EXPORTS_DIR, dataset=DATASET, experiment=EXPERIMENT, configuration=CONFIGURATION, seed=SEED)
    CHECKPOINT_PATH = None
    if read_json(RUN_DIR / "run_metadata.json").get("status") != "complete":
        CHECKPOINT_PATH = package_checkpoint(RUN_DIR, EXPORTS_DIR, dataset=DATASET, experiment=EXPERIMENT, configuration=CONFIGURATION, seed=SEED)
    SUMMARY_PATH = write_run_summary(EXPORTS_DIR / "run_summary.json", run_dir=RUN_DIR, commit_sha=COMMIT_SHA, seed=SEED, configuration=CONFIGURATION)
    print("ZIP:", ZIP_PATH)
    if CHECKPOINT_PATH: print("Checkpoint ZIP (upload as next session input):", CHECKPOINT_PATH)
    print("Summary:", SUMMARY_PATH)
